# Кандген на Instacart: семантические ID vs dense vs история + переключатель источников

Ноутбук самодостаточный: код записывается в файлы ячейками `%%writefile`.

**Как запустить на Kaggle**

1. *File → Import Notebook* — загрузи этот `.ipynb`.
2. Справа *Add Input* → вкладка *Competitions* → **Instacart Market Basket Analysis**
   (нужно один раз принять правила соревнования на его странице). Подойдёт и любой датасет-зеркало
   с теми же файлами (`orders.csv`, `order_products__prior.csv`, `order_products__train.csv`, `products.csv`,
   `aisles.csv`, `departments.csv`, можно `.csv.zip`) — ноутбук сам найдёт их в `/kaggle/input`.
3. *Settings → Accelerator → GPU T4 x2* (или P100). Интернет не нужен.
4. Проверка: `QUICK = True` → *Run All* (фейковые данные, пара минут).
   Полный прогон: `QUICK = False` → *Save Version → Save & Run All (Commit)*.
5. Результаты: `results.zip` во вкладке *Output* (`summary.md`, `sources.csv`, `union.csv`, `latency.csv`, `log.json`, лог).

Оценка времени на T4 при 30 тыс. пользователей: ~30–60 мин (не замер).


In [ ]:
import os, sys, time, subprocess, torch

REQUIRE_GPU = True
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет GPU")
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "Включи GPU: Settings → Accelerator"

BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
WORK = os.path.join(BASE, "candgen")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print("работаем в", WORK)
try:
    import tabulate  # noqa: F401
except ImportError:
    os.system(f"{sys.executable} -m pip install -q tabulate")


def run(args, log):
    """Запуск скрипта: вывод по ходу + лог (кроссплатформенный аналог `| tee`)."""
    print("$ python", args, flush=True)
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    with open(log, "w", encoding="utf-8") as f, subprocess.Popen(
            f'"{sys.executable}" -u {args}', shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", env=env) as p:
        for line in p.stdout:
            print(line, end="", flush=True)
            f.write(line)
    assert p.returncode == 0, f"процесс упал (код {p.returncode}), см. {log}"


## Код
Сгенерировано `kaggle/build_notebook.py` из исходников `candgen/`.

In [ ]:
%%writefile instacart.py
"""
Instacart: загрузка, фейковые данные того же формата (для локальных проверок) и протокол оценки.

Файлы Instacart Market Basket Analysis (Kaggle), можно и в .csv.zip:
  orders.csv                 order_id, user_id, eval_set, order_number, order_dow, order_hour_of_day, days_since_prior_order
  order_products__prior.csv  order_id, product_id, add_to_cart_order, reordered
  order_products__train.csv  то же
  products.csv               product_id, product_name, aisle_id, department_id
  aisles.csv, departments.csv

Особенности данных:
  * календарных дат нет — время восстанавливаем как сумму days_since_prior_order по заказам пользователя;
    значения обрезаны сверху на 30 («30» = «30 и больше»);
  * eval_set == "test" — заказы без состава (это тест соревнования), их выбрасываем.

Протокол (стандартный для next-basket на Instacart — глобального времени нет):
  test-цель = последний заказ пользователя, val-цель = предпоследний;
  модель учится на заказах до val-цели; история для теста включает val-заказ (модель не переобучается).
  Цель — множество различных товаров заказа; repeat — товар уже был в истории пользователя.
"""
import glob
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd

FILES = ["orders", "order_products__prior", "order_products__train", "products", "aisles", "departments"]


def find_root(path=None):
    """Папка, где лежит orders.csv(.zip): явный путь или поиск в /kaggle/input."""
    cands = [path] if path else []
    cands += sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/orders.csv*", recursive=True)})
    for c in cands:
        if c and (os.path.exists(os.path.join(c, "orders.csv")) or os.path.exists(os.path.join(c, "orders.csv.zip"))):
            return c
    raise FileNotFoundError("не нашёл orders.csv: укажи --data-root или подключи датасет Instacart в Kaggle")


def _read(root, name, **kw):
    for ext in (".csv", ".csv.zip"):
        p = os.path.join(root, name + ext)
        if os.path.exists(p):
            return pd.read_csv(p, **kw)
    raise FileNotFoundError(os.path.join(root, name + ".csv"))


def load_instacart(root, n_users=None, min_item_count=10, min_orders=4, seed=0):
    """
    Возвращает:
      events: DataFrame[user, item, order, t, pos] — user/item/order с 0, t — дни от первого заказа, pos — порядок в корзине;
      items:  DataFrame[item, product_id, name, aisle, department].
    n_users — случайная подвыборка пользователей (для скорости); min_item_count — порог частоты товара.
    """
    orders = _read(root, "orders", usecols=["order_id", "user_id", "eval_set", "order_number",
                                            "days_since_prior_order"])
    orders = orders[orders.eval_set != "test"]
    users = orders.user_id.unique()
    if n_users and n_users < len(users):
        users = np.random.default_rng(seed).choice(users, n_users, replace=False)
        orders = orders[orders.user_id.isin(users)]
    orders = orders.sort_values(["user_id", "order_number"])
    orders["t"] = orders.days_since_prior_order.fillna(0).groupby(orders.user_id).cumsum()

    cols = ["order_id", "product_id", "add_to_cart_order"]
    op = pd.concat([_read(root, "order_products__prior", usecols=cols),
                    _read(root, "order_products__train", usecols=cols)], ignore_index=True)
    op = op[op.order_id.isin(orders.order_id)]
    ev = op.merge(orders[["order_id", "user_id", "order_number", "t"]], on="order_id")

    cnt = ev.product_id.value_counts()
    ev = ev[ev.product_id.isin(cnt.index[cnt >= min_item_count])]
    n_ord = ev.groupby("user_id").order_number.nunique()
    ev = ev[ev.user_id.isin(n_ord.index[n_ord >= min_orders])]

    products = _read(root, "products").merge(_read(root, "aisles"), on="aisle_id") \
                                      .merge(_read(root, "departments"), on="department_id")
    pid = np.sort(ev.product_id.unique())
    item_of = pd.Series(np.arange(len(pid)), index=pid)
    ev = ev.assign(item=item_of[ev.product_id].to_numpy(),
                   user=ev.user_id.astype("category").cat.codes.to_numpy())
    ev["order"] = ev.groupby("user").order_number.rank(method="dense").astype(int) - 1
    events = ev.rename(columns={"add_to_cart_order": "pos"})[["user", "item", "order", "t", "pos"]] \
               .sort_values(["user", "order", "pos"]).reset_index(drop=True)
    items = products.set_index("product_id").loc[pid].reset_index()
    items = pd.DataFrame({"item": np.arange(len(pid)), "product_id": pid, "name": items.product_name.to_numpy(),
                          "aisle": items.aisle.to_numpy(), "department": items.department.to_numpy()})
    return events, items


# ----------------------------------------------------------------------------- фейковые данные

_ADJ = ["organic", "fresh", "low fat", "whole", "natural", "sparkling", "greek", "baby", "frozen", "roasted",
        "unsweetened", "vanilla", "spicy", "gluten free", "original", "family size", "mini", "classic"]


def make_fake_instacart(out_dir, n_users=3000, n_aisles=60, n_departments=12, products_per_aisle=40, seed=0):
    """
    Пишет CSV в формате Instacart. Структура похожа на настоящую: у пользователя любимые ряды (aisles),
    «свои» товары, которые он переказывает, и новые товары из любимых рядов; паузы между заказами 1–30 дней.
    Названия товаров собраны из слов ряда, так что текст несёт сигнал для семантических ID.
    """
    rng = np.random.default_rng(seed)
    os.makedirs(out_dir, exist_ok=True)
    aisle_names = [f"aisle{a} {w}" for a, w in zip(range(n_aisles), rng.choice(
        ["yogurt", "chips", "water", "bread", "cheese", "juice", "pasta", "soup", "tea", "coffee", "candy",
         "cereal", "eggs", "milk", "fruit", "vegetables", "frozen meals", "spices", "baby food", "pet food"], n_aisles))]
    aisle_dep = rng.integers(0, n_departments, n_aisles)
    rows, pid = [], 1
    for a in range(n_aisles):
        noun = aisle_names[a].split(" ", 1)[1]
        for _ in range(products_per_aisle):
            rows.append((pid, f"{' '.join(rng.choice(_ADJ, rng.integers(1, 3), replace=False))} {noun} {pid}",
                         a + 1, aisle_dep[a] + 1))
            pid += 1
    products = pd.DataFrame(rows, columns=["product_id", "product_name", "aisle_id", "department_id"])
    pop = rng.zipf(1.5, len(products)).clip(max=500).astype(float)
    by_aisle = [np.where(products.aisle_id.to_numpy() == a + 1)[0] for a in range(n_aisles)]

    orders, op, oid = [], [], 1
    for u in range(1, n_users + 1):
        fav_a = rng.choice(n_aisles, rng.integers(3, 9), replace=False)
        cand = np.concatenate([by_aisle[a] for a in fav_a])
        w = pop[cand] / pop[cand].sum()
        staples = set(rng.choice(cand, min(len(cand), rng.integers(3, 15)), replace=False, p=w).tolist())
        p_re = rng.uniform(0.3, 0.9)                      # насколько человек «рутинный»
        gap = rng.choice([3, 7, 14, 30])
        bought = set()
        for n in range(1, rng.integers(4, 40) + 1):
            orders.append((oid, u, "prior", n, None if n == 1 else int(min(30, max(0, rng.normal(gap, gap / 3))))))
            basket = [i for i in staples if rng.random() < p_re]
            basket += [i for i in bought if rng.random() < 0.05]
            basket += list(rng.choice(cand, rng.integers(1, 5), p=w))
            for k, i in enumerate(dict.fromkeys(basket)):
                op.append((oid, int(products.product_id[i]), k + 1, int(i in bought)))
                bought.add(i)
            oid += 1
    od = pd.DataFrame(orders, columns=["order_id", "user_id", "eval_set", "order_number", "days_since_prior_order"])
    od["order_dow"], od["order_hour_of_day"] = 0, 12
    last = od.groupby("user_id").order_number.transform("max") == od.order_number
    od.loc[last, "eval_set"] = "train"
    opd = pd.DataFrame(op, columns=["order_id", "product_id", "add_to_cart_order", "reordered"])
    tr = opd.order_id.isin(od.order_id[last])
    od.to_csv(os.path.join(out_dir, "orders.csv"), index=False)
    opd[~tr].to_csv(os.path.join(out_dir, "order_products__prior.csv"), index=False)
    opd[tr].to_csv(os.path.join(out_dir, "order_products__train.csv"), index=False)
    products.to_csv(os.path.join(out_dir, "products.csv"), index=False)
    pd.DataFrame({"aisle_id": range(1, n_aisles + 1), "aisle": aisle_names}).to_csv(
        os.path.join(out_dir, "aisles.csv"), index=False)
    pd.DataFrame({"department_id": range(1, n_departments + 1),
                  "department": [f"department {d}" for d in range(n_departments)]}).to_csv(
        os.path.join(out_dir, "departments.csv"), index=False)
    return out_dir


# ----------------------------------------------------------------------------- протокол

@dataclass
class Split:
    hist: dict        # user -> (items, times, orders): история до целевого заказа (по порядку)
    targets: dict     # user -> np.array различных товаров целевого заказа
    t_query: dict     # user -> время целевого заказа (момент запроса: пользователь открыл приложение)

    @property
    def users(self):
        return np.array(sorted(self.targets))


def make_protocol(events):
    """Возвращает (train_events, val, test). train_events — всё до val-заказа каждого пользователя."""
    last = events.groupby("user")["order"].transform("max")
    train = events[events.order < last - 1]

    def build(target_off):
        tgt_order = last - target_off
        hist_ev = events[events.order < tgt_order]
        tgt_ev = events[events.order == tgt_order]
        hist = {u: (g.item.to_numpy(), g.t.to_numpy(), g.order.to_numpy()) for u, g in hist_ev.groupby("user")}
        targets = {u: np.unique(g.item.to_numpy()) for u, g in tgt_ev.groupby("user") if u in hist}
        t_query = tgt_ev.groupby("user").t.first().to_dict()
        return Split(hist, targets, t_query)

    return train.reset_index(drop=True), build(1), build(0)


def item_counts(train, n_items):
    return np.bincount(train["item"], minlength=n_items)


In [ ]:
%%writefile semantic_ids.py
"""
Семантические ID (SID): товар → короткая последовательность токенов (c1, c2, ..., cL, d).

  1. Вектор товара:
       text — TF-IDF по «название | ряд | отдел» → TruncatedSVD → L2-норма (работает без интернета и внешних моделей);
       cf   — эмбеддинги товаров из обученной dense-модели (коллаборативные SID);
       text+cf — конкатенация нормированных векторов.
  2. Residual k-means (RQ-KMeans, как в OneRec): уровень l кластеризует остаток после уровней < l.
     Близкие товары делят префикс — на этом держится обобщение генеративного ретривала.
  3. Последний токен d — номер товара внутри группы с одинаковым (c1..cL), чтобы SID был уникален (как в TIGER).
"""
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer


def text_vectors(items, dim=64, seed=0):
    text = (items["name"].astype(str) + " | " + items["aisle"].astype(str) + " | "
            + items["department"].astype(str)).str.lower()
    X = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(text)
    dim = min(dim, X.shape[1] - 1)
    Z = TruncatedSVD(dim, random_state=seed).fit_transform(X)
    return Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)


def normed(Z):
    return Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)


def residual_kmeans(Z, codes=(128, 128, 128), seed=0):
    """Z: (N, d). Возвращает (N, L) коды и остаточную ошибку после каждого уровня."""
    R = Z.astype(np.float64).copy()
    out, err = [], []
    for l, k in enumerate(codes):
        k = min(k, len(R))
        km = MiniBatchKMeans(k, random_state=seed + l, batch_size=4096, n_init=3).fit(R)
        c = km.predict(R)
        R = R - km.cluster_centers_[c]
        out.append(c)
        err.append(float((R ** 2).sum(1).mean()))
    return np.stack(out, 1), err


def add_dedup(codes):
    """Добавляет токен-разделитель коллизий: номер товара внутри группы одинаковых кодов."""
    _, inv = np.unique(codes, axis=0, return_inverse=True)
    inv = inv.reshape(-1)
    order = np.argsort(inv, kind="stable")
    d = np.empty(len(inv), dtype=np.int64)
    first = np.r_[0, np.flatnonzero(np.diff(inv[order])) + 1]
    for s, e in zip(first, np.r_[first[1:], len(inv)]):
        d[order[s:e]] = np.arange(e - s)
    return np.concatenate([codes, d[:, None]], 1)


def build_sids(Z, codes=(128, 128, 128), seed=0):
    """Возвращает dict: sid (N, L+1), vocab (размер словаря на уровне), stats."""
    c, err = residual_kmeans(Z, codes, seed)
    sid = add_dedup(c)
    vocab = [int(sid[:, l].max()) + 1 for l in range(sid.shape[1])]
    groups = np.unique(c, axis=0, return_counts=True)[1]
    stats = {"levels": list(codes), "vocab": vocab, "unique codes (без d)": int(len(groups)),
             "доля товаров с коллизией": float((groups[groups > 1]).sum() / len(sid)),
             "макс. коллизия": int(groups.max()), "residual err по уровням": [round(e, 4) for e in err],
             "использовано кодов уровня 1": int(len(np.unique(c[:, 0])))}
    return {"sid": sid, "vocab": vocab, "stats": stats}


class Trie:
    """
    Допустимые продолжения префиксов для constrained beam search.
    Префикс уровня l кодируется числом key_l = key_{l-1} · V_l + c_l; допустимость проверяется
    бинарным поиском по отсортированным ключам реальных товаров.
    """

    def __init__(self, sid, vocab):
        self.vocab = vocab
        self.L = sid.shape[1]
        self.keys = []                                  # отсортированные ключи префиксов длины l+1
        key = np.zeros(len(sid), dtype=np.int64)
        for l in range(self.L):
            key = key * vocab[l] + sid[:, l]
            self.keys.append(np.unique(key))
        order = np.argsort(key)
        self.full_keys, self.item_of_key = key[order], order    # полный ключ → товар

    def key_extend(self, prefix_key, l, tokens):
        return prefix_key * self.vocab[l] + tokens

    def valid(self, keys, l):
        """keys: np.int64 любой формы — ключи префиксов длины l+1. Возвращает bool той же формы."""
        ks = self.keys[l]
        idx = np.clip(np.searchsorted(ks, keys), 0, len(ks) - 1)
        return ks[idx] == keys

    def items(self, full_keys):
        idx = np.clip(np.searchsorted(self.full_keys, full_keys), 0, len(self.full_keys) - 1)
        ok = self.full_keys[idx] == full_keys
        return np.where(ok, self.item_of_key[idx], -1)


In [ ]:
%%writefile models.py
"""
Модели кандгена: общий энкодер истории + две «головы» выдачи.

  DenseRetriever — dense-ретривал (SASRec/two-tower): вектор пользователя · эмбеддинги товаров, top-K точным MIPS.
                   Обучение — sampled softmax (общие равномерные негативы на батч) по всем товарам следующей корзины.
  SIDRetriever   — генеративный ретривал в духе TIGER: по состоянию пользователя авторегрессионно генерирует
                   токены семантического ID; constrained beam search по префиксному дереву (Trie) даёт top-K.
                   Вход энкодера — сумма эмбеддингов SID-токенов товара (обобщение через общие префиксы).

Запрос = история до целевого заказа + момент запроса (время целевого заказа).
"""
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

N_BUCKETS = 32


def time_bucket(dt_days):
    hours = torch.clamp(dt_days, min=0) * 24.0
    return torch.clamp(torch.floor(torch.log2(hours + 1.0) * 2).long(), 0, N_BUCKETS - 1)


class Encoder(nn.Module):
    """Трансформер над последовательностью товаров истории (корзины подряд); возвращает состояние на последней позиции."""

    def __init__(self, item_repr, d=64, n_layers=2, n_heads=2, max_len=100, dropout=0.1):
        super().__init__()
        self.item_repr = item_repr                     # callable: (B, L) item ids (+1, 0 = паддинг) → (B, L, d)
        self.pos = nn.Embedding(max_len, d)
        self.gap = nn.Embedding(N_BUCKETS, d)
        self.query = nn.Embedding(N_BUCKETS, d)
        layer = nn.TransformerEncoderLayer(d, n_heads, 4 * d, dropout, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(dropout)

    def forward(self, items, times, lens, t_query):
        B, L = items.shape
        prev = torch.cat([times[:, :1], times[:, :-1]], 1)
        pos = torch.arange(L, device=items.device).unsqueeze(0).expand(B, L)
        x = self.item_repr(items) + self.pos(pos) + self.gap(time_bucket(times - prev))
        causal = torch.triu(torch.ones(L, L, dtype=torch.bool, device=items.device), 1)
        h = self.norm(self.tf(self.drop(x), mask=causal))
        rows = torch.arange(B, device=items.device)
        last = h[rows, lens - 1]
        return last + self.query(time_bucket(t_query - times[rows, lens - 1]))


def _init(module):
    for m in module.modules():
        if isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)
            if m.padding_idx is not None:
                with torch.no_grad():
                    m.weight[m.padding_idx].zero_()


class DenseRetriever(nn.Module):
    def __init__(self, n_items, d=64, **enc_kw):
        super().__init__()
        self.n_items = n_items
        self.item_emb = nn.Embedding(n_items + 1, d, padding_idx=0)
        self.enc = Encoder(self.item_emb, d=d, **enc_kw)
        self.out = nn.Linear(d, d)
        _init(self)

    def user(self, items, times, lens, t_query):
        return self.out(self.enc(items, times, lens, t_query))

    def loss(self, batch, n_neg):
        """batch: items, times, lens, t_query, tgt_row (P,), tgt_item (P,) — пары (запрос, товар цели)."""
        items, times, lens, tq, row, tgt = batch
        u = self.user(items, times, lens, tq)                            # (B, d)
        neg = torch.randint(1, self.n_items + 1, (n_neg,), device=u.device)
        pos_logit = (u[row] * self.item_emb(tgt)).sum(-1)                # (P,)
        neg_logit = u @ self.item_emb(neg).T                             # (B, M)
        neg_logit = neg_logit[row].masked_fill(neg.unsqueeze(0) == tgt.unsqueeze(1), -1e9)
        logits = torch.cat([pos_logit.unsqueeze(1), neg_logit], 1)
        return -F.log_softmax(logits, 1)[:, 0].mean()

    @torch.no_grad()
    def scores(self, items, times, lens, t_query):
        """(B, n_items) — скоры по всем товарам (индексы товаров с 0)."""
        return self.user(items, times, lens, t_query) @ self.item_emb.weight[1:].T

    def item_vectors(self):
        return self.item_emb.weight[1:].detach().cpu().numpy()


class SIDRetriever(nn.Module):
    """
    Декодер: последовательность [u, e_0(c_0), ..., e_{T-2}(c_{T-2})] → на позиции t предсказывает c_t.
    u — состояние пользователя из энкодера; e_l — свой словарь токенов на каждом уровне.
    """

    def __init__(self, sid, vocab, d=64, dec_layers=2, n_heads=2, dropout=0.1, use_id=False, **enc_kw):
        super().__init__()
        self.T = sid.shape[1]
        self.vocab = vocab
        self.register_buffer("sid", torch.as_tensor(np.vstack([np.zeros((1, self.T), int), sid]), dtype=torch.long))
        self.tok = nn.ModuleList([nn.Embedding(v, d) for v in vocab])
        self.id_emb = nn.Embedding(len(sid) + 1, d, padding_idx=0) if use_id else None
        self.enc = Encoder(self._item_repr, d=d, **enc_kw)
        self.u_proj = nn.Linear(d, d)
        self.lvl = nn.Embedding(self.T, d)
        layer = nn.TransformerEncoderLayer(d, n_heads, 4 * d, dropout, batch_first=True, norm_first=True)
        self.dec = nn.TransformerEncoder(layer, dec_layers, enable_nested_tensor=False)
        self.dec_norm = nn.LayerNorm(d)
        self.heads = nn.ModuleList([nn.Linear(d, v) for v in vocab])
        _init(self)

    def _item_repr(self, items):
        codes = self.sid[items]                                          # (B, L, T)
        x = sum(self.tok[l](codes[..., l]) for l in range(self.T))
        x = x * (items != 0).unsqueeze(-1)
        if self.id_emb is not None:
            x = x + self.id_emb(items)
        return x

    def _decode(self, u, prefix):
        """u: (N, d); prefix: (N, l) токены уровней 0..l-1. Возвращает логиты уровня l: (N, V_l)."""
        l = prefix.shape[1]
        x = [self.u_proj(u).unsqueeze(1)] + [self.tok[j](prefix[:, j]).unsqueeze(1) for j in range(l)]
        x = torch.cat(x, 1) + self.lvl.weight[:l + 1].unsqueeze(0)
        causal = torch.triu(torch.ones(l + 1, l + 1, dtype=torch.bool, device=u.device), 1)
        h = self.dec_norm(self.dec(x, mask=causal))
        return self.heads[l](h[:, -1])

    def loss(self, batch, n_neg=None):
        items, times, lens, tq, row, tgt = batch
        u = self.enc(items, times, lens, tq)[row]                       # (P, d)
        codes = self.sid[tgt]                                            # (P, T)
        x = [self.u_proj(u).unsqueeze(1)] + [self.tok[j](codes[:, j]).unsqueeze(1) for j in range(self.T - 1)]
        x = torch.cat(x, 1) + self.lvl.weight.unsqueeze(0)
        causal = torch.triu(torch.ones(self.T, self.T, dtype=torch.bool, device=u.device), 1)
        h = self.dec_norm(self.dec(x, mask=causal))
        return sum(F.cross_entropy(self.heads[l](h[:, l]), codes[:, l]) for l in range(self.T)) / self.T

    @torch.no_grad()
    def beam_search(self, items, times, lens, t_query, trie_keys, beam):
        """
        Constrained beam search. trie_keys: список отсортированных тензоров ключей префиксов (на устройстве).
        Возвращает (full_keys (B, W), scores (B, W)) — по убыванию score; -inf у пустых лучей.
        """
        u = self.enc(items, times, lens, t_query)
        B, dev = u.shape[0], u.device
        keys = torch.zeros(B, 1, dtype=torch.long, device=dev)
        score = torch.zeros(B, 1, device=dev)
        prefix = torch.zeros(B, 1, 0, dtype=torch.long, device=dev)
        for l in range(self.T):
            W = keys.shape[1]
            V = self.vocab[l]
            logp = F.log_softmax(self._decode(u.repeat_interleave(W, 0), prefix.reshape(B * W, l)), -1)
            cand = keys.unsqueeze(-1) * V + torch.arange(V, device=dev)            # (B, W, V)
            ks = trie_keys[l]
            idx = torch.searchsorted(ks, cand.reshape(-1)).clamp_max(len(ks) - 1)
            ok = (ks[idx] == cand.reshape(-1)).reshape(B, W, V)
            total = (score.unsqueeze(-1) + logp.reshape(B, W, V)).masked_fill(~ok, -float("inf"))
            k = min(beam, W * V)
            score, flat = total.reshape(B, -1).topk(k, -1)
            parent, tok = flat // V, flat % V
            keys = cand.reshape(B, -1).gather(1, flat)
            prefix = torch.cat([prefix.gather(1, parent.unsqueeze(-1).expand(-1, -1, l)), tok.unsqueeze(-1)], -1)
        return keys, score


In [ ]:
%%writefile run.py
"""
Эксперимент: кандген для маркетплейса с повторами — Instacart.

Источники кандидатов (top-K, K до --kmax):
  pop          — популярность в train;
  hist         — товары из истории пользователя по частоте с затуханием по заказам (только повторы);
  hist+pop     — hist, остаток — популярное (классический бейзлайн);
  dense        — dense-ретривал (SASRec/two-tower, sampled softmax), точный MIPS по каталогу;
  dense-exp    — то же без товаров истории (источник «нового»);
  sid-<src>    — генеративный ретривал по семантическим ID (src: text — из текста товара, cf — из эмбеддингов dense);
  sid-<src>-exp, hybrid-<src> (SID-кандидаты → доранжирование dense-скором, как LIGER);
  union[...]   — K слотов: q из hist + (K − q) из источника «нового»; q — подобранная на val константа (fixed),
                 выученный переключатель на запрос (switch: регрессор доли повторов → q = α·K·p̂) или оракул.

Гипотезы: H1 — SID лучше на новом/хвосте, ID-модели и история — на повторах;
          H2 — переключатель между источниками на запрос лучше фиксированной квоты.

  python run.py --fake                               # дымовой прогон на фейковых данных формата Instacart
  python run.py --data-root <папка с csv> --n-users 30000
"""
import argparse
import copy
import json
import os
import random
import time

import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import HistGradientBoostingRegressor

from instacart import find_root, item_counts, load_instacart, make_fake_instacart, make_protocol
from models import DenseRetriever, SIDRetriever
from semantic_ids import Trie, build_sids, normed, text_vectors


# ----------------------------------------------------------------------------- батчи


def user_arrays(events):
    return {u: (g.item.to_numpy() + 1, g.t.to_numpy(), g.order.to_numpy()) for u, g in events.groupby("user")}


def train_samples(train, max_len, per_user):
    """Обучающие запросы: (история до заказа j, время заказа j, товары заказа j) для последних per_user заказов."""
    out = []
    for u, (it, ts, od) in user_arrays(train).items():
        bounds = np.flatnonzero(np.diff(od)) + 1                       # начала заказов 1..n-1
        for s in bounds[-per_user:]:
            e = np.searchsorted(od, od[s], side="right")
            h0 = max(0, s - max_len)
            out.append((it[h0:s], ts[h0:s], ts[s], np.unique(it[s:e])))
    return out


def collate(samples, max_len, device):
    B = len(samples)
    items = np.zeros((B, max_len), np.int64)
    times = np.zeros((B, max_len), np.float32)
    lens = np.zeros(B, np.int64)
    tq = np.zeros(B, np.float32)
    rows, tgts = [], []
    for b, (it, ts, t, tg) in enumerate(samples):
        n = len(it)
        items[b, :n], times[b, :n], times[b, n:], lens[b], tq[b] = it, ts, ts[-1], n, t
        if tg is not None:
            rows.append(np.full(len(tg), b))
            tgts.append(tg)
    T = lambda a: torch.from_numpy(a).to(device)
    batch = [T(items), T(times), T(lens), T(tq)]
    if rows:
        batch += [T(np.concatenate(rows)), T(np.concatenate(tgts).astype(np.int64))]
    return batch


def query_samples(split, users, max_len):
    return [((split.hist[u][0][-max_len:] + 1), split.hist[u][1][-max_len:], split.t_query[u], None) for u in users]


# ----------------------------------------------------------------------------- метрики


def recall_table(ranked, users, split, head, ks):
    """ranked: (U, ≥K) id товаров (−1 — пусто). Recall по пользователям + микро по repeat / explore / explore-хвост."""
    out = {}
    tg = [split.targets[u] for u in users]
    rep = [np.isin(t, split.hist[u][0]) for t, u in zip(tg, users)]
    tail = [~head[t] for t in tg]
    for k in ks:
        hit = [np.isin(t, ranked[r, :k]) for r, t in enumerate(tg)]
        n = lambda m: max(sum(x.sum() for x in m), 1)
        out[f"R@{k}"] = float(np.mean([h.mean() for h in hit]))
        out[f"R@{k} repeat"] = float(sum((h & r).sum() for h, r in zip(hit, rep)) / n(rep))
        out[f"R@{k} explore"] = float(sum((h & ~r).sum() for h, r in zip(hit, rep)) / n([~r for r in rep]))
        out[f"R@{k} explore-tail"] = float(sum((h & ~r & tl).sum() for h, r, tl in zip(hit, rep, tail))
                                           / n([~r & tl for r, tl in zip(rep, tail)]))
    return out


def recall_at(ranked, users, split, k):
    return float(np.mean([np.isin(split.targets[u], ranked[r, :k]).mean() for r, u in enumerate(users)]))


# ----------------------------------------------------------------------------- источники


def pad(lists, k):
    out = np.full((len(lists), k), -1, np.int64)
    for r, l in enumerate(lists):
        l = l[:k]
        out[r, :len(l)] = l
    return out


def hist_rank(split, users, decay=0.9):
    """История пользователя: Σ decay^(заказов назад) по покупкам товара."""
    res = []
    for u in users:
        it, _, od = split.hist[u]
        w = decay ** (od.max() - od)
        s = pd.Series(w).groupby(it).sum().sort_values(ascending=False)
        res.append(s.index.to_numpy())
    return res


def fill(primary, filler, k):
    """primary (списки) + остаток из filler (общий ранжированный массив), без повторов."""
    out = []
    for p in primary:
        seen = set(p.tolist())
        rest = [i for i in filler[:k + len(p)] if i not in seen][:max(0, k - len(p))]
        out.append(np.concatenate([p[:k], np.array(rest, np.int64)]))
    return pad(out, k)


def topk_scores(scores, k, exclude=None):
    s = scores.copy()
    if exclude is not None:
        for r, ex in enumerate(exclude):
            s[r, ex] = -np.inf
    idx = np.argpartition(-s, k, 1)[:, :k]
    order = np.argsort(-np.take_along_axis(s, idx, 1), 1)
    idx = np.take_along_axis(idx, order, 1)
    return np.where(np.isfinite(np.take_along_axis(s, idx, 1)), idx, -1)


@torch.no_grad()
def dense_rank(model, split, users, args, k, exclude_hist=False, cands=None):
    """
    Батчами, без матрицы (U, N) в памяти: top-k dense-скоров (exclude_hist — без товаров истории)
    или, если заданы cands (U, M), — доранжирование этих кандидатов dense-скором.
    """
    model.eval()
    outs = []
    for s in range(0, len(users), args.eval_batch):
        us = users[s:s + args.eval_batch]
        b = collate(query_samples(split, us, args.max_len), args.max_len, args.device)
        sc = model.scores(*b).float()
        if cands is not None:
            c = torch.as_tensor(cands[s:s + len(us)], device=sc.device)
            cs = sc.gather(1, c.clamp_min(0)).masked_fill(c < 0, -float("inf"))
            order = cs.argsort(1, descending=True)
            outs.append(c.gather(1, order).cpu().numpy())
            continue
        if exclude_hist:
            for r, u in enumerate(us):
                sc[r, torch.as_tensor(np.unique(split.hist[u][0]), device=sc.device)] = -float("inf")
        val, idx = sc.topk(k, 1)
        outs.append(torch.where(torch.isfinite(val), idx, -1).cpu().numpy())
    return np.concatenate(outs)


@torch.no_grad()
def sid_beam(model, trie, trie_keys, split, users, args, beam):
    """(U, beam) товаров по убыванию лог-вероятности SID."""
    model.eval()
    outs = []
    for s in range(0, len(users), args.beam_batch):
        b = collate(query_samples(split, users[s:s + args.beam_batch], args.max_len), args.max_len, args.device)
        keys, score = model.beam_search(*b, trie_keys, beam)
        items = trie.items(keys.cpu().numpy())
        items[~np.isfinite(score.cpu().numpy())] = -1
        outs.append(pad(list(items), beam))
    return np.concatenate(outs)


def drop_hist(ranked, split, users, k):
    return pad([r[(r >= 0) & ~np.isin(r, split.hist[u][0])] for r, u in zip(ranked, users)], k)


def union(hist_lists, exp_ranked, q, k):
    q = np.broadcast_to(np.asarray(q), (len(hist_lists),))
    out = []
    for r, h in enumerate(hist_lists):
        m = int(min(q[r], len(h), k))
        e = exp_ranked[r][exp_ranked[r] >= 0][:k - m]
        out.append(np.concatenate([h[:m], e]))
    return pad(out, k)


# ----------------------------------------------------------------------------- переключатель


def user_features(split, users):
    rows = []
    for u in users:
        it, ts, od = split.hist[u]
        bounds = np.r_[0, np.flatnonzero(np.diff(od)) + 1, len(od)]
        seen, rep_share = set(), []
        for s, e in zip(bounds[:-1], bounds[1:]):
            b = set(it[s:e].tolist())
            if seen:
                rep_share.append(len(b & seen) / len(b))
            seen |= b
        gaps = np.diff(ts[bounds[:-1]]) if len(bounds) > 2 else np.array([0.0])
        rows.append([len(bounds) - 1, len(seen), len(it) / (len(bounds) - 1),
                     np.mean(rep_share) if rep_share else 0.0, rep_share[-1] if rep_share else 0.0,
                     split.t_query[u] - ts[-1], gaps.mean(), gaps.std(), bounds[-1] - bounds[-2]])
    return np.array(rows, float)


def repeat_frac(split, users):
    return np.array([np.isin(split.targets[u], split.hist[u][0]).mean() for u in users])


def fit_switch(train_events, users_seed):
    """Регрессор доли повторов в следующей корзине: обучается на последнем заказе train (история — до него)."""
    _, _, pseudo = make_protocol(train_events)          # цель = последний train-заказ, история — до него
    us = pseudo.users
    if len(us) > 20000:
        us = np.random.default_rng(users_seed).choice(us, 20000, replace=False)
    reg = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=users_seed)
    reg.fit(user_features(pseudo, us), repeat_frac(pseudo, us))
    return reg


# ----------------------------------------------------------------------------- обучение


def train_model(model, samples, val, val_users, args, eval_fn, name):
    opt = torch.optim.Adam(model.parameters(), lr=args.lr)
    best, best_state, best_ep, bad = -1.0, None, 0, 0
    for ep in range(args.max_epochs):
        model.train()
        random.shuffle(samples)
        tot, n, t0 = 0.0, 0, time.time()
        for s in range(0, len(samples), args.batch):
            loss = model.loss(collate(samples[s:s + args.batch], args.max_len, args.device), args.n_neg)
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot, n = tot + loss.item(), n + 1
        rec = eval_fn(model)
        print(f"  [{name}] epoch {ep + 1} loss {tot / n:.4f} val R@50 {rec:.4f} ({time.time() - t0:.0f} с)", flush=True)
        if rec > best + 1e-4:
            best, best_state, best_ep, bad = rec, copy.deepcopy(model.state_dict()), ep + 1, 0
        else:
            bad += 1
            if bad >= args.patience:
                break
    model.load_state_dict(best_state)
    return model, best_ep, best


# ----------------------------------------------------------------------------- латентность


def sync(args):
    if str(args.device).startswith("cuda"):
        torch.cuda.synchronize()


def online_ms(fn, users, n=100, warmup=10):
    ms = []
    for i, u in enumerate(users[:n + warmup]):
        t = time.perf_counter()
        fn(np.array([u]))
        if i >= warmup:
            ms.append(1000 * (time.perf_counter() - t))
    return {f"online p{q} ms": float(np.percentile(ms, q)) for q in (50, 95, 99)}


# ----------------------------------------------------------------------------- main


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", default=None)
    ap.add_argument("--fake", action="store_true", help="сгенерировать фейковые данные формата Instacart")
    ap.add_argument("--fake-users", type=int, default=3000)
    ap.add_argument("--n-users", type=int, default=30000)
    ap.add_argument("--min-item-count", type=int, default=10)
    ap.add_argument("--max-len", type=int, default=100)
    ap.add_argument("--per-user", type=int, default=5, help="обучающих заказов на пользователя")
    ap.add_argument("--dim", type=int, default=64)
    ap.add_argument("--layers", type=int, default=2)
    ap.add_argument("--batch", type=int, default=256)
    ap.add_argument("--eval-batch", type=int, default=256)
    ap.add_argument("--beam-batch", type=int, default=32)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--n-neg", type=int, default=2048)
    ap.add_argument("--max-epochs", type=int, default=30)
    ap.add_argument("--patience", type=int, default=3)
    ap.add_argument("--sid-codes", type=int, nargs="+", default=[128, 128, 128])
    ap.add_argument("--sid-sources", nargs="+", default=["text", "cf"], choices=["text", "cf", "text+cf"])
    ap.add_argument("--kmax", type=int, default=200)
    ap.add_argument("--ks", type=int, nargs="+", default=[10, 50, 100, 200])
    ap.add_argument("--val-users", type=int, default=3000, help="пользователей val для ранней остановки")
    ap.add_argument("--head-share", type=float, default=0.2, help="доля самых популярных товаров = «голова»")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="results/main")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = ap.parse_args()
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    os.makedirs(args.out, exist_ok=True)
    K, ks = args.kmax, [k for k in args.ks if k <= args.kmax]
    log = {"args": vars(args)}

    # --- данные
    t0 = time.time()
    if args.fake:
        root = make_fake_instacart(os.path.join(args.out, "fake_data"), n_users=args.fake_users, seed=args.seed)
    else:
        root = find_root(args.data_root)
    events, items = load_instacart(root, n_users=args.n_users, min_item_count=args.min_item_count, seed=args.seed)
    train, val, test = make_protocol(events)
    N = len(items)
    cnt = item_counts(train, N)
    head = np.zeros(N, bool)
    head[np.argsort(-cnt)[:max(1, int(args.head_share * N))]] = True
    vu_all, tu = val.users, test.users
    vu = vu_all if len(vu_all) <= args.val_users else np.random.default_rng(args.seed).choice(vu_all, args.val_users, replace=False)
    rf = repeat_frac(test, tu)
    log["data"] = {"root": root, "events": len(events), "users": int(events.user.nunique()), "items": N,
                   "train events": len(train), "test users": len(tu),
                   "test basket size": float(np.mean([len(test.targets[u]) for u in tu])),
                   "repeat share (test, по товарам)": float(np.concatenate(
                       [np.isin(test.targets[u], test.hist[u][0]) for u in tu]).mean()),
                   "distinct history items (медиана)": float(np.median([len(np.unique(test.hist[u][0])) for u in tu])),
                   "load sec": round(time.time() - t0)}
    print(json.dumps(log["data"], ensure_ascii=False, indent=1), flush=True)
    samples = train_samples(train, args.max_len, args.per_user)
    print(f"обучающих запросов: {len(samples)}", flush=True)
    enc_kw = dict(n_layers=args.layers, max_len=args.max_len)

    R = {}                                                     # имя → ранжированный массив (U_test, K)
    lat = []

    # --- простые источники
    pop = np.argsort(-cnt)
    hist_t = hist_rank(test, tu)
    R["pop"] = np.tile(pop[:K], (len(tu), 1))
    R["hist"] = pad(hist_t, K)
    R["hist+pop"] = fill(hist_t, pop, K)

    # --- dense
    t0 = time.time()
    dense = DenseRetriever(N, d=args.dim, **enc_kw).to(args.device)
    dense, ep, _ = train_model(dense, list(samples), val, vu, args,
                               lambda m: recall_at(dense_rank(m, val, vu, args, 50), vu, val, 50), "dense")
    log["dense"] = {"best epoch": ep, "train sec": round(time.time() - t0)}
    R["dense"] = dense_rank(dense, test, tu, args, K)
    R["dense-exp"] = dense_rank(dense, test, tu, args, K, exclude_hist=True)

    # --- семантические ID и генеративный ретривал
    vecs = {}
    for src in args.sid_sources:
        if src == "text":
            vecs[src] = text_vectors(items, dim=64, seed=args.seed)
        elif src == "cf":
            vecs[src] = normed(dense.item_vectors())
        else:
            vecs[src] = normed(np.hstack([text_vectors(items, 64, args.seed), normed(dense.item_vectors())]))
    log["sid"] = {}
    sid_models = {}
    for src, Z in vecs.items():
        t0 = time.time()
        S = build_sids(Z, tuple(args.sid_codes), seed=args.seed)
        trie = Trie(S["sid"], S["vocab"])
        tk = [torch.as_tensor(k, device=args.device) for k in trie.keys]
        m = SIDRetriever(S["sid"], S["vocab"], d=args.dim, **enc_kw).to(args.device)
        m, ep, _ = train_model(m, list(samples), val, vu, args,
                               lambda mm: recall_at(sid_beam(mm, trie, tk, val, vu, args, 50), vu, val, 50), f"sid-{src}")
        log["sid"][src] = {**S["stats"], "best epoch": ep, "train sec": round(time.time() - t0)}
        sid_models[src] = (m, trie, tk)
        beam = sid_beam(m, trie, tk, test, tu, args, 2 * K)
        R[f"sid-{src}"] = beam[:, :K]
        R[f"sid-{src}-exp"] = drop_hist(beam, test, tu, K)
        R[f"hybrid-{src}"] = dense_rank(dense, test, tu, args, K, cands=beam)[:, :K]   # LIGER-подобный гибрид
        print(f"  sid-{src}: {json.dumps(S['stats'], ensure_ascii=False)}", flush=True)

    # --- объединение источников: фиксированная квота vs переключатель vs оракул
    reg = fit_switch(train, args.seed)
    p_val, p_test = reg.predict(user_features(val, vu_all)).clip(0, 1), reg.predict(user_features(test, tu)).clip(0, 1)
    rf_val = repeat_frac(val, vu_all)
    log["switch"] = {"corr(p̂, доля повторов) val": float(np.corrcoef(p_val, rf_val)[0, 1]),
                     "corr test": float(np.corrcoef(p_test, rf)[0, 1])}
    hist_v = hist_rank(val, vu_all)
    exp_sources = {"dense-exp": None}
    exp_sources.update({f"sid-{s}-exp": s for s in sid_models})
    union_rows = []
    for name, src in exp_sources.items():
        if src is None:
            ev = dense_rank(dense, val, vu_all, args, K, exclude_hist=True)
        else:
            m, trie, tk = sid_models[src]
            ev = drop_hist(sid_beam(m, trie, tk, val, vu_all, args, 2 * K), val, vu_all, K)
        et = R[name]
        for k in ks:
            grid_q = sorted({int(round(k * f)) for f in np.linspace(0, 1, 11)})
            q_best = max(grid_q, key=lambda q: recall_at(union(hist_v, ev, q, k), vu_all, val, k))
            grid_a = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0]
            qa = lambda p, a: np.round(np.minimum(1.0, a * p) * k)
            a_best = max(grid_a, key=lambda a: recall_at(union(hist_v, ev, qa(p_val, a), k), vu_all, val, k))
            for kind, q in (("fixed", q_best), ("switch", qa(p_test, a_best)), ("oracle", qa(rf, 1.0)),
                            ("hist-first", k)):
                r = recall_table(union(hist_t, et, q, k), tu, test, head, [k])
                union_rows.append({"exp source": name, "K": k, "kind": kind,
                                   "param": q_best if kind == "fixed" else (a_best if kind == "switch" else ""),
                                   **{c.replace(f"@{k}", "@K"): v for c, v in r.items()}})
    un = pd.DataFrame(union_rows)

    # --- метрики источников
    rows = [{"source": n, **recall_table(r, tu, test, head, ks)} for n, r in R.items()]
    for n, r in R.items():
        rows[[x["source"] for x in rows].index(n)]["coverage@100"] = len(np.unique(r[:, :100][r[:, :100] >= 0])) / N
    res = pd.DataFrame(rows).set_index("source")

    # --- латентность (batch=1, полный путь до top-K)
    one = argparse.Namespace(**{**vars(args), "eval_batch": 1, "beam_batch": 1})

    def timed_online(fn):
        def wrap(u):
            fn(u)
            sync(args)
        return online_ms(wrap, tu)

    lat.append({"source": "hist+pop", **timed_online(lambda u: fill(hist_rank(test, u), pop, K))})
    lat.append({"source": "dense", **timed_online(lambda u: dense_rank(dense, test, u, one, K))})
    for src, (m, trie, tk) in sid_models.items():
        lat.append({"source": f"sid-{src} (beam {K})",
                    **timed_online(lambda u, m=m, trie=trie, tk=tk: sid_beam(m, trie, tk, test, u, one, K))})
        lat.append({"source": f"hybrid-{src} (beam {2 * K} + dense)",
                    **timed_online(lambda u, m=m, trie=trie, tk=tk: dense_rank(
                        dense, test, u, one, K, cands=sid_beam(m, trie, tk, test, u, one, 2 * K)))})
    lat = pd.DataFrame(lat).set_index("source")
    for name, m in [("dense", dense)] + [(f"sid-{s}", v[0]) for s, v in sid_models.items()]:
        log.setdefault("params", {})[name] = sum(p.numel() for p in m.parameters())

    # --- отчёт
    res.to_csv(os.path.join(args.out, "sources.csv"))
    un.to_csv(os.path.join(args.out, "union.csv"), index=False)
    lat.to_csv(os.path.join(args.out, "latency.csv"))
    with open(os.path.join(args.out, "log.json"), "w", encoding="utf-8") as f:
        json.dump(log, f, ensure_ascii=False, indent=1, default=str)
    md = lambda df, **kw: df.to_markdown(**kw) if _has_tabulate() else "```\n" + df.to_string() + "\n```"
    k0 = 100 if 100 in ks else ks[-1]
    lines = [f"# Кандген на Instacart ({'фейк' if args.fake else 'реальные данные'}), устройство {args.device}\n",
             "## Данные\n", "```json\n" + json.dumps(log["data"], ensure_ascii=False, indent=1) + "\n```\n",
             "## Семантические ID\n", "```json\n" + json.dumps(log["sid"], ensure_ascii=False, indent=1) + "\n```\n",
             f"## Источники: Recall@K (тест)\n", md(res[[f"R@{k}" for k in ks]], floatfmt=".4f"),
             f"\n\n## Разбивка при K={k0}: повторы / новое / новое из хвоста\n",
             md(res[[f"R@{k0}", f"R@{k0} repeat", f"R@{k0} explore", f"R@{k0} explore-tail", "coverage@100"]],
                floatfmt=".4f"),
             "\n\n## Объединение источников: hist + источник нового\n",
             "`fixed` — квота q подобрана на val; `switch` — q = α·K·p̂ (p̂ — регрессор доли повторов, α на val); "
             "`oracle` — q = K·(истинная доля повторов); `hist-first` — вся история, затем новое.\n",
             "```json\n" + json.dumps(log["switch"], ensure_ascii=False) + "\n```\n",
             md(un.set_index(["exp source", "K", "kind"]), floatfmt=".4f"),
             f"\n\n## Латентность (batch=1, {args.device})\n", md(lat, floatfmt=".2f"),
             "\n\n## Обучение\n", "```json\n" + json.dumps({"dense": log["dense"], "params": log["params"]},
                                                           ensure_ascii=False, indent=1) + "\n```\n"]
    text = "\n".join(lines)
    with open(os.path.join(args.out, "summary.md"), "w", encoding="utf-8") as f:
        f.write(text)
    print("\n" + text)


def _has_tabulate():
    try:
        import tabulate  # noqa: F401
        return True
    except ImportError:
        return False


if __name__ == "__main__":
    main()


## Прогон

In [ ]:
# ---- настройки ----
QUICK = False              # True — фейковые данные формата Instacart, пара минут
N_USERS = 30000              # подвыборка пользователей Instacart (всего ~206 тыс.)
SID_CODES = "128 128 128"    # размеры кодбуков residual k-means по уровням
SID_SOURCES = "text cf"      # text — SID из текста товара, cf — из эмбеддингов dense-модели
KMAX, KS = 200, "10 50 100 200"
COMMON = f"--n-users {N_USERS} --sid-codes {SID_CODES} --sid-sources {SID_SOURCES} --kmax {KMAX} --ks {KS}"
if QUICK:
    COMMON = ("--fake --fake-users 1200 --n-users 1200 --min-item-count 5 --max-epochs 2 --per-user 3 "
              "--val-users 500 --sid-codes 32 32 32 --kmax 100 --ks 10 50 100")
os.makedirs("results", exist_ok=True)
print(COMMON)


In [ ]:
t0 = time.time()
run(f"run.py {COMMON} --out results/main", "results/main.log")
print(f"готово за {(time.time() - t0) / 60:.1f} мин")


## Результаты

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("results/main/summary.md", encoding="utf-8").read()))


In [ ]:
import shutil
for junk in ("results/main/fake_data",):
    shutil.rmtree(junk, ignore_errors=True)
print("сохранено:", shutil.make_archive(os.path.join(BASE, "results"), "zip", "results"))
